# Web Crawler Demo: CNN with Callbacks (Memory-Efficient)

This notebook demonstrates the WebCrawler Spider using callbacks with `accumulate_results=False` to process results as they're crawled without accumulating them in memory.

**Comparison:** Compare memory usage with `crawl_cnn.ipynb` which accumulates all results in memory.

In [1]:
import json
import logging
from collections import Counter
from datetime import datetime

import psutil

from WebCrawler import Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

# Get initial memory usage
process = psutil.Process()
initial_memory = process.memory_info().rss / 1024 / 1024  # MB
print(f"Initial memory: {initial_memory:.2f} MB")

Initial memory: 94.27 MB


## Stream URLs with Callbacks (Memory-Efficient)

Define callbacks to process and display results as they're crawled, without keeping them in memory.

In [2]:
# Track crawl statistics
crawl_stats = {
    "total_pages": 0,
    "total_internal_links": 0,
    "total_external_links": 0,
    "failed_urls": [],
}

# Store first 10 URLs for display
urls_crawled = []


def on_page_crawled(doc):
    """Called after each page is crawled - process and discard."""
    crawl_stats["total_pages"] += 1
    crawl_stats["total_internal_links"] += len(doc.internal_links)
    crawl_stats["total_external_links"] += len(doc.external_links)

    # Store first 10 URLs for display
    if len(urls_crawled) < 10:
        urls_crawled.append(
            {
                "url": doc.url,
                "title": doc.title[:60],
                "status": doc.status_code,
                "internal_links": len(doc.internal_links),
                "external_links": len(doc.external_links),
            }
        )

    # Print progress every 10 pages
    if crawl_stats["total_pages"] % 10 == 0:
        current_memory = process.memory_info().rss / 1024 / 1024
        print(
            f"  Crawled {crawl_stats['total_pages']} pages | "
            f"Memory: {current_memory:.2f} MB | "
            f"Last: {doc.url}"
        )

    # Don't return anything - results are discarded
    return None


def on_error(url, exception):
    """Called when a page fails to crawl."""
    crawl_stats["failed_urls"].append({"url": url, "error": str(exception)})
    print(f"  ❌ Failed: {url}")


def on_crawl_complete():
    """Called when crawl finishes."""
    print(f"\n✓ Crawl complete!")


# Run spider with callbacks - accumulate_results=False means no memory buildup
spider = Spider(
    start_url="https://www.cnn.com",
    max_depth=3,
    on_page_crawled=on_page_crawled,
    on_error=on_error,
    on_crawl_complete=on_crawl_complete,
    accumulate_results=False,  # KEY: Don't accumulate in memory
    show_progress=True,
)

print(f"Starting streaming crawl with max_depth=3...\n")
result = await spider.run_async()

print(f"\nResult list: {len(result)} (empty because accumulate_results=False)")

2026-06-08 15:25:35,189 - WebCrawler.Spider - DEBUG - Spider initialized: strategy=BFS, max_depth=3


Starting streaming crawl with max_depth=3...



Crawling: 0 URLs [00:00, ? URLs/s]2026-06-08 15:25:35,528 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com (attempt 1)
2026-06-08 15:25:35,528 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com (attempt 1)
2026-06-08 15:25:35,615 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-08 15:25:35,615 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com (Total Visited: 1)
Crawling: 1 URLs [00:00,  2.47 URLs/s, visited=1, pending=0]2026-06-08 15:25:35,626 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/us (attempt 1)
2026-06-08 15:25:35,626 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/us (attempt 1)
2026-06-08 15:25:35,657 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/business (attempt 1)
2026-06-08 15:25:35,657 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/business (attempt 1)
2026-06-08 15:25:35,659 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/entertainment (attempt 1)
2026-06-08 15:2

  Crawled 10 pages | Memory: 396.78 MB | Last: https://www.cnn.com/politics


2026-06-08 15:25:36,256 WebCrawler.Spider INFO     Visited: https://www.cnn.com/science (Total Visited: 14)
2026-06-08 15:25:36,256 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/science (Total Visited: 14)
Crawling: 14 URLs [00:01, 17.61 URLs/s, visited=14, pending=5910]2026-06-08 15:25:36,257 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/account/settings (attempt 1)
2026-06-08 15:25:36,257 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/account/settings (attempt 1)
2026-06-08 15:25:36,300 WebCrawler.Spider INFO     Visited: https://www.cnn.com/weather (Total Visited: 15)
2026-06-08 15:25:36,300 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/weather (Total Visited: 15)
Crawling: 15 URLs [00:01, 17.61 URLs/s, visited=15, pending=6786]2026-06-08 15:25:36,328 WebCrawler.Spider INFO     Visited: https://www.cnn.com/climate (Total Visited: 16)
2026-06-08 15:25:36,328 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/climate (Total Visited:

...

2026-06-08 15:29:55,495 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/05/14/business/video/why-japanese-chip-bags-going-black-white-digvid (Total Visited: 1631)
2026-06-08 15:29:55,495 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/2026/05/14/business/video/why-japanese-chip-bags-going-black-white-digvid (Total Visited: 1631)
Crawling: 1631 URLs [04:20,  2.81 URLs/s, visited=1631, pending=6576228]2026-06-08 15:29:56,817 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/2026/06/02/investing/stocks-memory-ai-boom (attempt 1)
2026-06-08 15:29:56,817 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/2026/06/02/investing/stocks-memory-ai-boom (attempt 1)
2026-06-08 15:29:56,837 WebCrawler.Spider DEBUG    Fetching https://www.cnn.com/2026/06/01/economy/finding-new-job-challenges (attempt 1)
2026-06-08 15:29:56,837 - WebCrawler.Spider - DEBUG - Fetching https://www.cnn.com/2026/06/01/economy/finding-new-job-challenges (attempt 1)
2026-06-08 15:29:57,13

  Crawled 1640 pages | Memory: 7236.39 MB | Last: https://www.cnn.com/2026/03/27/investing/us-stocks-iran


2026-06-08 15:30:02,463 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/04/15/investing/us-stocks-iran-war (Total Visited: 1641)
2026-06-08 15:30:02,463 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/2026/04/15/investing/us-stocks-iran-war (Total Visited: 1641)
Crawling: 1641 URLs [04:27,  2.73 URLs/s, visited=1641, pending=6630086]2026-06-08 15:30:02,729 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/04/02/investing/us-stocks-iran (Total Visited: 1642)
2026-06-08 15:30:02,729 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/2026/04/02/investing/us-stocks-iran (Total Visited: 1642)
Crawling: 1642 URLs [04:27,  2.98 URLs/s, visited=1642, pending=6635956]2026-06-08 15:30:02,982 WebCrawler.Spider INFO     Visited: https://www.cnn.com/2026/03/20/investing/gold-price-drop-fed-rate-iran (Total Visited: 1643)
2026-06-08 15:30:02,982 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com/2026/03/20/investing/gold-price-drop-fed-rate-iran (Total


✓ Crawl complete!


CancelledError: 

## Crawl Statistics

In [ ]:
# Calculate final memory usage
final_memory = process.memory_info().rss / 1024 / 1024
memory_change = final_memory - initial_memory

print("=" * 70)
print("CRAWL STATISTICS (Streaming with Callbacks)")
print("=" * 70)
print(f"Total pages crawled: {crawl_stats['total_pages']}")
print(f"Total internal links found: {crawl_stats['total_internal_links']}")
print(f"Total external links found: {crawl_stats['total_external_links']}")
print(f"Failed URLs: {len(crawl_stats['failed_urls'])}")
print()
print(f"Initial memory: {initial_memory:.2f} MB")
print(f"Final memory: {final_memory:.2f} MB")
print(f"Memory change: {memory_change:+.2f} MB")
print(f"Avg memory per page: {memory_change / crawl_stats['total_pages']:.4f} MB")
print()
print(
    f"Avg links per page: "
    f"{(crawl_stats['total_internal_links'] + crawl_stats['total_external_links']) / crawl_stats['total_pages']:.1f}"
)

## First 10 URLs Crawled

In [ ]:
print(f"{'URL':<50} {'Title':<20} {'Status':<8} {'Links':<8}")
print("-" * 90)

for item in urls_crawled:
    url_short = item["url"][:49]
    title_short = item["title"][:19]
    total_links = item["internal_links"] + item["external_links"]
    print(
        f"{url_short:<50} {title_short:<20} "
        f"{item['status']:<8} {total_links:<8}"
    )

## Failed URLs (if any)

In [ ]:
if crawl_stats["failed_urls"]:
    print(f"Failed URLs ({len(crawl_stats['failed_urls'])}):")
    for failed in crawl_stats["failed_urls"]:
        print(f"  • {failed['url']}")
        print(f"    Error: {failed['error'][:60]}...")
else:
    print("No failed URLs!")

## Comparison: Streaming vs Accumulating

**This notebook (Streaming):**
- ✅ Results processed as pages are crawled
- ✅ No results kept in memory
- ✅ Predictable, minimal memory growth
- ✅ Perfect for large crawls

**crawl_cnn.ipynb (Accumulating):**
- ❌ All pages accumulated in `documents` list
- ❌ Memory grows linearly with page count
- ❌ Can cause OOM on very large crawls
- ✅ Convenient for analysis at end

**For large crawls (1000+ pages):** Use streaming callbacks (`accumulate_results=False`)